# Part B — 05 Hyperparameter Tuning

This notebook improves the two strongest predictive models:

- Logistic Regression
- Random Forest

Hyperparameters are selected using the 2023 validation data.

The 2024 patent data remains untouched and will only be used once for
final model evaluation.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    PredefinedSplit
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

ROOT = Path.cwd().parent

PROCESSED_DIR = ROOT / "Data" / "processed"
TABLES_DIR = ROOT / "Outputs" / "tables"

DATA_PATH = (
    PROCESSED_DIR /
    "01_part_b_target_dataset.parquet"
)

df = pd.read_parquet(DATA_PATH)

print("Rows:", f"{len(df):,}")
print("Columns:", df.shape[1])

Rows: 49,556
Columns: 26


In [2]:
numeric_features = [
    "inventor_count",
    "assignee_count",
    "cpc_count",
    "backward_citation_count",
    "title_word_count",
    "abstract_word_count",
    "claims_text_length",
    "claims_word_count",
    "filing_to_grant_days"
]

categorical_features = [
    "cpc_group",
    "assignee_country"
]

feature_columns = (
    numeric_features +
    categorical_features
)

X = df[feature_columns].copy()
y = df["high_impact"].copy()

In [3]:
tuning_mask = df["grant_year"].isin(
    [2021, 2022, 2023]
)

X_tuning = X.loc[tuning_mask].copy()
y_tuning = y.loc[tuning_mask].copy()

tuning_years = (
    df.loc[tuning_mask, "grant_year"]
    .copy()
)

print(
    "Tuning dataset:",
    X_tuning.shape
)

print(
    tuning_years.value_counts()
    .sort_index()
)

Tuning dataset: (44062, 11)
grant_year
2021    11246
2022    15219
2023    17597
Name: count, dtype: int64


In [4]:
test_fold = np.where(
    tuning_years == 2023,
    0,
    -1
)

temporal_split = PredefinedSplit(
    test_fold=test_fold
)

print(
    "Number of validation splits:",
    temporal_split.get_n_splits()
)

Number of validation splits: 1


In [5]:
numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            numeric_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ]
)

In [6]:
logistic_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=3000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

In [7]:
logistic_params = {
    "classifier__C": [
        0.01,
        0.1,
        1,
        10,
        100
    ]
}

In [8]:
logistic_search = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=logistic_params,
    scoring="roc_auc",
    cv=temporal_split,
    n_jobs=-1,
    refit=True
)

logistic_search.fit(
    X_tuning,
    y_tuning
)

print(
    "Best Logistic parameters:",
    logistic_search.best_params_
)

print(
    "Best validation ROC-AUC:",
    round(
        logistic_search.best_score_,
        4
    )
)

Best Logistic parameters: {'classifier__C': 0.1}
Best validation ROC-AUC: 0.6367


In [9]:
rf_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            RandomForestClassifier(
                class_weight="balanced",
                random_state=42,
                n_jobs=1
            )
        )
    ]
)

In [11]:
rf_params = {
    "classifier__n_estimators": [
        200,
        300,
        500
    ],

    "classifier__max_depth": [
        None,
        10,
        20,
        30
    ],

    "classifier__min_samples_split": [
        2,
        5,
        10
    ],

    "classifier__min_samples_leaf": [
        1,
        2,
        4
    ],

    "classifier__max_features": [
        "sqrt",
        "log2"
    ]
}

In [12]:
rf_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_params,
    n_iter=20,
    scoring="roc_auc",
    cv=temporal_split,
    random_state=42,
    n_jobs=-1,
    refit=True,
    verbose=1
)

rf_search.fit(
    X_tuning,
    y_tuning
)

Fitting 1 folds for each of 20 candidates, totalling 20 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'classifier__max_depth': [None, 10, ...], 'classifier__max_features': ['sqrt', 'log2'], 'classifier__min_samples_leaf': [1, 2, ...], 'classifier__min_samples_split': [2, 5, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.","PredefinedSpl...ape=(44062,)))"
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than ma

In [13]:
print(
    "Best Random Forest parameters:"
)

print(
    rf_search.best_params_
)

print(
    "\nBest validation ROC-AUC:",
    round(
        rf_search.best_score_,
        4
    )
)

Best Random Forest parameters:
{'classifier__n_estimators': 200, 'classifier__min_samples_split': 10, 'classifier__min_samples_leaf': 4, 'classifier__max_features': 'log2', 'classifier__max_depth': 20}

Best validation ROC-AUC: 0.6568


In [14]:
best_logistic_params = {
    key.replace(
        "classifier__",
        ""
    ): value
    for key, value
    in logistic_search.best_params_.items()
}

best_rf_params = {
    key.replace(
        "classifier__",
        ""
    ): value
    for key, value
    in rf_search.best_params_.items()
}

print(
    "Logistic:",
    best_logistic_params
)

print(
    "\nRandom Forest:",
    best_rf_params
)

Logistic: {'C': 0.1}

Random Forest: {'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': 'log2', 'max_depth': 20}


In [15]:
train_mask = df["grant_year"].isin(
    [2021, 2022]
)

val_mask = (
    df["grant_year"] == 2023
)

X_train = X.loc[train_mask].copy()
y_train = y.loc[train_mask].copy()

X_val = X.loc[val_mask].copy()
y_val = y.loc[val_mask].copy()

In [16]:
tuned_logistic = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            LogisticRegression(
                **best_logistic_params,
                max_iter=3000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

In [17]:
tuned_rf = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            RandomForestClassifier(
                **best_rf_params,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [18]:
tuned_models = {
    "Tuned Logistic Regression":
        tuned_logistic,

    "Tuned Random Forest":
        tuned_rf
}

tuned_results = []

for name, model in tuned_models.items():

    print(
        f"Training {name}..."
    )

    model.fit(
        X_train,
        y_train
    )

    pred = model.predict(
        X_val
    )

    prob = model.predict_proba(
        X_val
    )[:, 1]

    tuned_results.append({
        "Model": name,

        "Accuracy":
            accuracy_score(
                y_val,
                pred
            ),

        "Precision":
            precision_score(
                y_val,
                pred
            ),

        "Recall":
            recall_score(
                y_val,
                pred
            ),

        "F1":
            f1_score(
                y_val,
                pred
            ),

        "ROC_AUC":
            roc_auc_score(
                y_val,
                prob
            ),

        "PR_AUC":
            average_precision_score(
                y_val,
                prob
            )
    })

Training Tuned Logistic Regression...
Training Tuned Random Forest...


In [19]:
tuned_results_df = pd.DataFrame(
    tuned_results
)

tuned_results_df.round(3)

,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
0,Tuned Logistic Regression,0.563,0.274,0.662,0.388,0.637,0.307
1,Tuned Random Forest,0.642,0.307,0.566,0.398,0.657,0.334


In [20]:
TUNING_RESULTS_PATH = (
    TABLES_DIR /
    "part_b_tuned_model_results.csv"
)

tuned_results_df.to_csv(
    TUNING_RESULTS_PATH,
    index=False
)

print(
    "Saved:",
    TUNING_RESULTS_PATH
)

Saved: /Users/janakdobariya/Bramha/NLP_Engineering/BA/ai_patent_business_analytics/Outputs/tables/part_b_tuned_model_results.csv
